# Bot Activity and Trade Inspection

Inspect the trading database for bot activity, open positions, and recent closed positions.

In [3]:
import sqlite3
from pathlib import Path

db_path = Path('data') / 'trading.db'
assert db_path.exists(), f'No DB found at {db_path}'
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

print('TABLES:')
for row in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    print(' -', row['name'])

print('\nBOT_ACTIVITY SCHEMA:')
for row in cur.execute("PRAGMA table_info(bot_activity)"):
    print(dict(row))

print('\nSAMPLE BOT_ACTIVITY ROWS:')
for row in cur.execute("SELECT ts, symbol, action, detail FROM bot_activity ORDER BY ts DESC LIMIT 20"):
    print(row['ts'], row['symbol'], row['action'], row['detail'][:200])

print('\nOPEN POSITIONS:')
for row in cur.execute("SELECT id, symbol, side, qty, entry_price, stop_loss, take_profit, status, pnl FROM paper_positions WHERE status='open' ORDER BY id"):
    print(dict(row))

print('\nPAPER_POSITIONS SCHEMA:')
for row in cur.execute("PRAGMA table_info(paper_positions)"):
    print(dict(row))

print('\nRECENT CLOSED POSITIONS:')
for row in cur.execute("SELECT id, symbol, side, qty, entry_price, exit_price, stop_loss, take_profit, pnl, closed_at FROM paper_positions WHERE status='closed' ORDER BY closed_at DESC LIMIT 20"):
    print(dict(row))
conn.close()

TABLES:
 - account_state
 - ai_analyses
 - backtest_runs
 - bot_activity
 - coin_playbook
 - coin_sentiment
 - equity_snapshots
 - ohlcv
 - paper_orders
 - paper_positions
 - paper_trades
 - sqlite_sequence

BOT_ACTIVITY SCHEMA:
{'cid': 0, 'name': 'id', 'type': 'INTEGER', 'notnull': 0, 'dflt_value': None, 'pk': 1}
{'cid': 1, 'name': 'ts', 'type': 'INTEGER', 'notnull': 1, 'dflt_value': None, 'pk': 0}
{'cid': 2, 'name': 'symbol', 'type': 'TEXT', 'notnull': 1, 'dflt_value': "''", 'pk': 0}
{'cid': 3, 'name': 'action', 'type': 'TEXT', 'notnull': 1, 'dflt_value': None, 'pk': 0}
{'cid': 4, 'name': 'detail', 'type': 'TEXT', 'notnull': 1, 'dflt_value': "'{}'", 'pk': 0}

SAMPLE BOT_ACTIVITY ROWS:
1784134143637 SHIBUSDT scalp_skip {"reason": "dust: notional $0.00 < $1.00", "explanation": "Skipped SHIBUSDT: 10% of equity works out to $0.00, below the $1.00 minimum."}
1784134142713 ARBUSDT scalp_skip {"reason": "dust: notional $0.00 < $1.00", "explanation": "Skipped ARBUSDT: 10% of equity works out

In [ ]:
import json

conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

print('\nBOT_ACTIVITY ENTER/EXIT DETAILS:')
for row in cur.execute(
    "SELECT ts, symbol, action, detail FROM bot_activity WHERE action IN ('enter','exit','reject','skip','halt','cycle_end') ORDER BY ts DESC LIMIT 50"
):
    detail = row['detail']
    try:
        parsed = json.loads(detail)
        detail_str = json.dumps(parsed, indent=None)
    except Exception:
        detail_str = detail
    print(row['ts'], row['symbol'], row['action'], detail_str)

conn.close()